In [5]:
# ============================================================
# PARTE 1: DICE por clase (promedio del dataset) — versión ligera
# ============================================================
import json, math, numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
from tqdm import tqdm
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation



In [3]:
DIR_MODA = Path("modanet")

DIR_IMAGENES = DIR_MODA / "images"
DIR_LABELS = DIR_MODA / "annotations-seg"
DIR_PRED_DINO = DIR_MODA / "pred_dino"
DIR_PRED_SEGFORMER = DIR_MODA / "pred_segformer"



!mkdir {DIR_MODA}
!wget "https://www.dropbox.com/scl/fo/g0or9rc44z7ey8oj8b5gv/AJnHmMLpYvYNYj-DK8Vf5oA/annotations-seg.tar?rlkey=l61rkqa63s9kv57bh1tp1oiie&dl=0" -q -O annotations-seg.tar
!wget "https://www.dropbox.com/scl/fo/g0or9rc44z7ey8oj8b5gv/AMvjsHNsh406QE1R7fhEhG8/images_modanet.tar?rlkey=l61rkqa63s9kv57bh1tp1oiie&dl=0" -q -O images_modanet.tar

!tar -xf annotations-seg.tar -C {DIR_MODA}
!tar -xf images_modanet.tar -C {DIR_MODA}



In [6]:
import random

IM_DIR = DIR_IMAGENES
JSON_DIR = DIR_LABELS


IMG_EXTS = (".jpg", ".jpeg", ".png")
json_files = sorted(JSON_DIR.glob("*.json"))

sampled_json_files = random.sample(json_files, 10)

In [7]:
class_names = [
    "background","bag","belt","boots","footwear","outer",
    "dress","sunglasses","pants","top","shorts","skirt",
    "headwear","scarf & tie".replace(" ","")  # evita espacios en nombre
]
NUM_CLASSES = len(class_names)

# ---------- MAPEOS CORREGIDOS ----------
map_pred_modelId_to_final = {
    0:0, 1:12, 2:0, 3:7, 4:9, 5:11, 6:8, 7:6, 8:2, 9:4, 10:4,
    11:0, 12:0, 13:0, 14:0, 15:0, 16:1, 17:13
}
map_gt_jsonId_to_final = {i:i for i in range(NUM_CLASSES)}  # identidad (ajusta si tu JSON usa otros ids)

In [8]:
# ---------- MODELO (usa CPU si tu PC es modesto) ----------


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

SEG_CKPT_PRED = "mattmdjaga/segformer_b2_clothes"
seg_iproc = SegformerImageProcessor.from_pretrained(SEG_CKPT_PRED)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_CKPT_PRED).to(device).eval()

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:417: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/109M [00:00<?, ?B/s]

In [9]:
# ---------- UTILIDADES ----------
def load_annotations(json_path: Path):
    with open(json_path, 'r') as f:
        anns = json.load(f)
    for a in anns:
        if isinstance(a.get('category_id', None), str):
            try: a['category_id'] = int(a['category_id'])
            except: pass
    return anns

def find_image_for_json(jp: Path) -> Path:
    stem = jp.stem
    for ext in IMG_EXTS:
        p = IM_DIR / f"{stem}{ext}"
        if p.exists(): return p
    raise FileNotFoundError(f"No encontré imagen para {jp.name}")

def polygons_to_mask(img_size_hw, anns, background_id=0):
    H, W = img_size_hw
    mask = Image.new('I', (W, H), color=background_id)
    draw = ImageDraw.Draw(mask)
    for ann in anns:
        cat = int(ann.get('category_id', background_id))
        for poly in ann.get('segmentation', []):
            if not poly: continue
            pts = list(zip(poly[0::2], poly[1::2]))
            draw.polygon(pts, fill=cat)
    return np.array(mask, dtype=np.int64)

def apply_map(mask_np: np.ndarray, mapping: dict, ignore_value: int = -1):
    out = np.full_like(mask_np, ignore_value)
    for k, v in mapping.items():
        out[mask_np == k] = v
    return out

def segformer_predict_ids_final(image_pil: Image.Image) -> np.ndarray:
    """Devuelve máscara en ids FINALES (ModaNet) usando el mapeo de predicción."""
    H, W = image_pil.size[1], image_pil.size[0]
    inputs = seg_iproc(images=image_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        out = seg_model(**inputs)  # logits [1,C,h',w']
    logits_up = F.interpolate(out.logits, size=(H, W), mode="bilinear", align_corners=False)
    pred_model = logits_up.argmax(1).squeeze(0).cpu().numpy().astype(np.int32)  # 0..17
    pred_final = apply_map(pred_model, map_pred_modelId_to_final, ignore_value=-1)
    pred_final[pred_final < 0] = 0  # lo no mapeado -> fondo
    return pred_final

def dice_per_class_image(gt_final: np.ndarray, pred_final: np.ndarray, K: int):
    """Dice solo en clases presentes en el GT (las ausentes quedan NaN)."""
    scores = np.full((K,), np.nan, dtype=np.float32)
    for c in range(K):
        gt_c = (gt_final == c)
        if gt_c.sum() == 0:
            continue
        pred_c = (pred_final == c)
        inter = np.logical_and(gt_c, pred_c).sum()
        total = gt_c.sum() + pred_c.sum()
        scores[c] = (2.0 * inter / total) if total > 0 else 0.0
    return scores

def plot_bars(values, labels, title="Dice por clase (promedio)"):
    x = np.arange(len(values))
    plt.figure(figsize=(10,5))
    plt.bar(x, values)
    plt.xticks(x, labels, rotation=45, ha='right')
    plt.ylim(0,1)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [10]:
# ---------- DRIVER (procesa solo las primeras N imágenes) ----------
def run_partial_dice(MAX_IMAGES=10):
    dice_list = []
    files = json_files[:MAX_IMAGES]
    for jp in tqdm(files, desc=f"Dice sobre {len(files)} imágenes"):
        # 1) Imagen + GT desde JSON
        img_path = find_image_for_json(jp)
        img = Image.open(img_path).convert("RGB")
        anns = load_annotations(jp)
        H, W = img.size[1], img.size[0]
        gt_json = polygons_to_mask((H, W), anns, background_id=0)            # ids JSON
        gt_final = apply_map(gt_json, map_gt_jsonId_to_final, ignore_value=-1) # ids finales
        gt_final[gt_final < 0] = 0

        # 2) Predicción del modelo → ids finales
        pred_final = segformer_predict_ids_final(img)

        # 3) Dice por imagen (solo clases presentes en GT)
        dice_img = dice_per_class_image(gt_final, pred_final, NUM_CLASSES)
        dice_list.append(dice_img)

    dice_avg = np.nanmean(np.vstack(dice_list), axis=0)  # promedio ignorando NaN
    print("\nDice promedio por clase:")
    for name, val in zip(class_names, dice_avg):
        print(f"{name:12s}: {val:.3f}")
    plot_bars(dice_avg, class_names, title=f"Dice por clase (promedio {len(files)} imgs)")

In [5]:
if device = 'cpu' then:
  run_partial_dice(MAX_IMAGES=len(sampled_json_files))
else:
  run_partial_dice(MAX_IMAGES=len(json_files))

NameError: name 'run_partial_dice' is not defined

In [ ]:
import os, math, shutil, subprocess, sys
import torch
import umap
import torch.nn.functional as F
import numpy as np
import torchvision.transforms as T

from sklearn.preprocessing import StandardScaler
from transformers import SegformerModel

# --- Config: descarga en el directorio de trabajo actual
DINO_REPO_DIR     = "./dinov3"
DINO_HUB_ENTRY    = "dinov3_vitb16"   # opciones comunes: dinov3_vitb16, dinov3_vitl14, etc.
DINO_WEIGHTS_PATH = "./dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"  # cambia si ya tienes otro archivo
GDOWN_FILE_ID     = "1Fznrc_pDwp7iaUBhAoWUKPI1vsGpHy8m"               # el que compartiste

In [ ]:
def _run(cmd):
    """Ejecuta un comando de forma silenciosa; lanza excepción si falla."""
    print(f"[setup] {' '.join(cmd)}")
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

def ensure_dinov3_assets(try_download=True):
    """
    Asegura que el repo ./dinov3 y los pesos DINO_WEIGHTS_PATH existan en el cwd de Colab.
    - Si faltan, clona el repo y descarga pesos con gdown (si está permitido).
    """
    # Repo
    if not os.path.isdir(DINO_REPO_DIR):
        if try_download:
            # git está en Colab por defecto
            _run(["git", "clone", "https://github.com/facebookresearch/dinov3.git", DINO_REPO_DIR])
        else:
            raise FileNotFoundError(f"No se encontró {DINO_REPO_DIR} y try_download=False")

    # Pesos
    if not os.path.isfile(DINO_WEIGHTS_PATH):
        if not try_download:
            print(f"[setup] Pesos no encontrados en {DINO_WEIGHTS_PATH} (try_download=False). "
                  f"Colócalos en esa ruta y vuelve a ejecutar.")
        else:
            # Asegurar gdown
            if shutil.which("gdown") is None:
                _run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            try:
                _run(["gdown", "--id", GDOWN_FILE_ID, "--output", DINO_WEIGHTS_PATH])
            except subprocess.CalledProcessError:
                print("[setup] No se pudieron descargar los pesos con gdown. "
                      "Si ya los tienes, colócalos en", DINO_WEIGHTS_PATH)

def load_dino_small(device="cpu", try_download=True):
    """
    Carga DINOv3 (ViT-B/16 por defecto) desde el repo local ./dinov3 y pesos en el cwd.
    Devuelve (modelo.eval().to(device), preproc_transform)
    """
    # Prepro oficial típica para DINOv3 (518 px, mean/std ImageNet)
    pre = T.Compose([
        T.Resize((518, 518)),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

    # Asegurar assets en el directorio actual
    ensure_dinov3_assets(try_download=try_download)

    # Carga vía torch.hub local (source='local')
    try:
        model = torch.hub.load(
            DINO_REPO_DIR,
            DINO_HUB_ENTRY,
            source="local",
            weights=DINO_WEIGHTS_PATH
        )
        model = model.to(device).eval()
        return model, pre
    except Exception as e:
        print("[DINOv3] Error al cargar vía torch.hub local:", e)
        return None, pre

def get_vit_tokens(model, img_t):
    """
    Extrae los patch tokens de la última capa como [B, P, C] (sin CLS),
    más (ph, pw) y C para reconstruir la rejilla.
    Funciona con diferentes variantes de la API de DINOv3.
    """
    with torch.no_grad():
        # 1) Vía get_intermediate_layers (si existe)
        if hasattr(model, 'get_intermediate_layers'):
            out = model.get_intermediate_layers(img_t, n=1)[0]  # [B, 1+P, C] o [B, P, C]
            B, N, C = out.shape
            # Detectar si hay CLS
            # Si N-1 es cuadrado perfecto => el primero es CLS
            if int(math.isclose(math.sqrt(max(N-1, 0)), round(math.sqrt(max(N-1, 0))))) and (N-1) > 0:
                ph = pw = int(round(math.sqrt(N-1)))
                tokens = out[:, 1:, :]  # quitar CLS
            else:
                # Asume que ya vienen sin CLS
                ph = pw = int(round(math.sqrt(N)))
                tokens = out
            return tokens, (ph, pw), C

        # 2) Vía forward_features con return_all_tokens=True (timm-like / dinov3)
        if hasattr(model, 'forward_features'):
            try:
                out = model.forward_features(img_t, return_all_tokens=True)
            except TypeError:
                # Algunos forward_features no aceptan ese kwarg
                out = model.forward_features(img_t)

            # Normalmente out puede ser un dict con claves tipo:
            # 'x_norm_patchtokens', 'x_norm_clstoken', etc., o un tensor
            if isinstance(out, dict):
                if 'x_norm_patchtokens' in out:
                    tokens = out['x_norm_patchtokens']  # [B, P, C]
                    B, P, C = tokens.shape
                    ph = pw = int(round(math.sqrt(P)))
                    return tokens, (ph, pw), C
                elif 'tokens' in out:
                    tokens = out['tokens']  # [B, N, C] (posible CLS)
                else:
                    # Buscar tensor más grande
                    candidates = [v for v in out.values() if isinstance(v, torch.Tensor) and v.ndim == 3]
                    if not candidates:
                        raise RuntimeError("forward_features dict no contiene tokens 3D.")
                    # Heurística: elegir el de mayor N
                    tokens = max(candidates, key=lambda t: t.shape[1])
            else:
                # out es un tensor
                tokens = out  # [B, N, C]

            B, N, C = tokens.shape
            # Detectar CLS con cuadrado perfecto
            if int(math.isclose(math.sqrt(max(N-1, 0)), round(math.sqrt(max(N-1, 0))))) and (N-1) > 0:
                ph = pw = int(round(math.sqrt(N-1)))
                tokens = tokens[:, 1:, :]  # quitar CLS
            else:
                ph = pw = int(round(math.sqrt(N)))
            return tokens, (ph, pw), C

        # 3) Fallback: hook en el último bloque para capturar tokens
        last_tokens = {}
        def _hook(_, __, output):
            last_tokens['x'] = output

        # Buscar un bloque "blocks" o similar
        blocks = getattr(model, 'blocks', None) or getattr(getattr(model, 'model', None), 'blocks', None)
        if blocks is None or len(blocks) == 0:
            raise RuntimeError("No se encontraron 'blocks' para registrar hook de tokens.")

        h = blocks[-1].register_forward_hook(_hook)
        try:
            _ = model(img_t)
            if 'x' not in last_tokens:
                raise RuntimeError("Hook no capturó tokens.")
            out = last_tokens['x']  # [B, N, C]
            B, N, C = out.shape
            if int(math.isclose(math.sqrt(max(N-1, 0)), round(math.sqrt(max(N-1, 0))))) and (N-1) > 0:
                ph = pw = int(round(math.sqrt(N-1)))
                tokens = out[:, 1:, :]
            else:
                ph = pw = int(round(math.sqrt(N)))
                tokens = out
            return tokens, (ph, pw), C
        finally:
            h.remove()

        # Si nada funcionó:
        raise RuntimeError("Este DINOv3 no expone una ruta conocida para obtener tokens.")


In [ ]:
def segformer_encoder_and_tokens(ckpt=SEG_CKPT_PRED, device="cpu", image_pil=None):
    iproc = SegformerImageProcessor.from_pretrained(ckpt)
    enc   = SegformerModel.from_pretrained(ckpt).to(device).eval()
    inputs = iproc(images=image_pil, return_tensors="pt")
    px = inputs["pixel_values"].to(device)
    with torch.no_grad():
        out = enc(pixel_values=px)
    feats = out.last_hidden_state  # [1,C,ph,pw]
    B,C,ph,pw = feats.shape
    tokens = feats.permute(0,2,3,1).reshape(B, ph*pw, C)  # [1,P,C]
    return tokens, (ph,pw), C

def region_means(token_grid, grid_hw, gt_final_np, K):
    ph,pw = grid_hw
    C = token_grid.shape[-1]
    mask_t = torch.from_numpy(gt_final_np[None,None,...]).float()
    mask_small = F.interpolate(mask_t, size=(ph,pw), mode='nearest').long().squeeze()
    toks = token_grid[0].reshape(ph,pw,C)
    embs, labels = [], []
    for c in range(K):
        sel = (mask_small==c)
        if sel.any():
            embs.append(toks[sel].mean(0).cpu().numpy())
            labels.append(c)
    return (np.vstack(embs) if embs else np.zeros((0,C))), labels

def run_umap_on_vectors(embs, labels, title):
    if len(embs)==0:
        print("Sin embeddings.")
        return
    X = StandardScaler().fit_transform(embs)
    Y = umap.UMAP(n_components=2, metric="cosine", random_state=42).fit_transform(X)
    plt.figure(figsize=(7,6))
    for c in sorted(set(labels)):
        idx = [i for i,cc in enumerate(labels) if cc==c]
        plt.scatter(Y[idx,0], Y[idx,1], s=12, alpha=0.85, label=class_names[c])
    plt.title(title); plt.legend(bbox_to_anchor=(1.04,1), loc="upper left"); plt.grid(True, linestyle='--', alpha=0.3); plt.tight_layout(); plt.show()

def run_partial_embeddings_umap(MAX_IMAGES=6, device="cpu"):
    dino_model, dino_pre = load_dino_small(device)
    dino_vecs, dino_labs = [], []
    seg_vecs,  seg_labs  = [], []

    for jp in tqdm(json_files[:MAX_IMAGES], desc="Embeddings (parcial)"):
        img_path = find_image_for_json(jp)
        img = Image.open(img_path).convert("RGB")
        anns = load_annotations(jp)
        H, W = img.size[1], img.size[0]
        gt_json  = polygons_to_mask((H,W), anns, background_id=0)
        gt_final = apply_map(gt_json, map_gt_jsonId_to_final, ignore_value=-1)
        gt_final[gt_final < 0] = 0

        # DINO
        if dino_model is not None:
            try:
                img_t = dino_pre(img).unsqueeze(0).to(device)
                tok, hw, _ = get_vit_tokens(dino_model, img_t)
                e,l = region_means(tok, hw, gt_final, NUM_CLASSES)
                if len(l)>0: dino_vecs.append(e); dino_labs += l
            except Exception as e:
                print("DINO error:", e)

        # SegFormer encoder
        try:
            tok, hw, _ = segformer_encoder_and_tokens(ckpt=SEG_CKPT_PRED, device=device, image_pil=img)
            e,l = region_means(tok, hw, gt_final, NUM_CLASSES)
            if len(l)>0: seg_vecs.append(e); seg_labs += l
        except Exception as e:
            print("SegFormer encoder error:", e)

    if dino_vecs:
        D = np.vstack(dino_vecs)
        run_umap_on_vectors(D, dino_labs, "UMAP — DINO (parcial)")
    if seg_vecs:
        S = np.vstack(seg_vecs)
        run_umap_on_vectors(S, seg_labs,  "UMAP — SegFormer (parcial)")

In [ ]:
if device = 'cpu' then:
  run_partial_embeddings_umap(MAX_IMAGES=len(sampled_json_files), device=device)
else:
  run_partial_embeddings_umap(MAX_IMAGES=len(json_files), device=device)